# Text-to-query com plano tipado

Fixture local identificada: nenhuma chamada ao LLM. O modelo real teria de produzir o mesmo tipo de plano antes da validação.


In [ ]:
import sys, tempfile
from pathlib import Path
raiz = Path.cwd()
if raiz.name == 'notebooks': raiz = raiz.parent
sys.path.insert(0, str(raiz / 'src'))
from secretaria.banco import Banco, Contexto
from secretaria.consulta_estruturada import executar_plano, PlanoConsultaMensagens


In [ ]:
pergunta = 'Quantas mensagens de Bruno mencionam orçamento e Alfa?'
plano_fixture = PlanoConsultaMensagens(tipo='contagem', autor='Bruno', termos=['orçamento', 'Alfa'])
with tempfile.TemporaryDirectory() as pasta:
    with Banco(Path(pasta) / 'demo.sqlite3') as banco:
        banco.preparar_demo()
        contexto = Contexto('demo', 'ana')
        corpus_id = banco.selecionar_corpus(contexto, 'Demonstração fictícia')
        banco.importar(contexto, corpus_id, raiz / 'dados/ficticios/conversa_empresa.txt')
        resultado = executar_plano(banco, contexto, corpus_id, plano_fixture)
        print('Pergunta:', pergunta)
        print('Plano (FIXTURE LOCAL):', resultado['plano'])
        print('SQL permitido:', resultado['sql_executado'])
        print('Parâmetros:', resultado['parametros_sql'])
        print('Resultado:', resultado['total'])
        assert resultado['total'] == 1


A contagem percorre as mensagens importadas autorizadas. Ela não calcula saldo nem comprova recebimento. Na aplicação, a OpenAI gera o plano tipado; Python continua sendo responsável por validar e executar o SELECT.
